# 12.8 工具调用强化学习深挖

> 🕐 预估学习时间：40分钟

SFT 能让模型“会写工具 JSON”，但何时调用、调错如何恢复，更适合用可验证奖励做 RL（Toolformer/API-Bank/τ-bench 思路）。

深挖点：
- 动作空间：call / answer / clarify
- 轨迹奖励：成功、步数、非法调用
- 拒学“乱调工具”
- 与过程监督结合


## 1. 迷你工具环境

工具：`calc`、`search`。任务：算术或事实查询。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(0)
random.seed(0)

TOOLS = ['calc', 'search', 'answer']


def env_step(task, action, arg):
    if action == 'calc':
        try:
            # only a+b
            a, b = map(int, arg.split('+'))
            return str(a + b), 0.0, False
        except Exception:
            return 'err', -0.5, False
    if action == 'search':
        kb = {'capital france': 'paris', 'capital germany': 'berlin'}
        return kb.get(arg, 'unknown'), 0.0, False
    if action == 'answer':
        ok = (arg.strip().lower() == task['gold'].lower())
        return 'done', 1.0 if ok else -1.0, True
    return 'bad', -1.0, True


def make_task():
    if random.random() < 0.5:
        a, b = random.randint(1, 9), random.randint(1, 9)
        return {'q': f'{a}+{b}', 'gold': str(a + b), 'type': 'calc'}
    item = random.choice([('capital france', 'paris'), ('capital germany', 'berlin')])
    return {'q': item[0], 'gold': item[1], 'type': 'search'}


print(env_step({'gold': '3'}, 'calc', '1+2'))
print(env_step({'gold': 'paris'}, 'search', 'capital france'))
print('Key: Verifiable environments turn tool use into RL with grounded rewards.')


## 2. 策略网络与 REINFORCE

状态 = 任务类型嵌入 + 是否已有观测；输出动作分布。


In [ ]:
class ToolPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(3, 16)  # task type id
        self.fc = nn.Linear(16 + 2, 3)  # + has_obs, obs_ok

    def forward(self, task_type, has_obs, obs_ok):
        x = torch.cat([self.emb(task_type), has_obs, obs_ok], dim=-1)
        return self.fc(x)


def run_episode(policy, task, greedy=False):
    typ = torch.tensor([0 if task['type'] == 'calc' else 1])
    has_obs = torch.zeros(1, 1)
    obs_ok = torch.zeros(1, 1)
    obs = None
    logps = []
    reward = 0.0
    for step in range(3):
        logits = policy(typ, has_obs, obs_ok)
        dist = torch.distributions.Categorical(logits=logits)
        act = dist.probs.argmax() if greedy else dist.sample()
        logps.append(dist.log_prob(act))
        name = TOOLS[int(act)]
        if name == 'calc':
            arg = task['q'] if task['type'] == 'calc' else '0+0'
        elif name == 'search':
            arg = task['q']
        else:
            arg = obs if obs is not None else 'idk'
        obs, r, done = env_step(task, name, arg)
        reward += r - 0.05  # step cost
        has_obs = torch.ones(1, 1)
        obs_ok = torch.tensor([[0.0 if obs in {'err', 'unknown', 'bad'} else 1.0]])
        if done:
            break
    return reward, logps


policy = ToolPolicy()
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)
print('=== REINFORCE Tool Policy ===')
for step in range(120):
    task = make_task()
    R, logps = run_episode(policy, task)
    loss = -R * torch.stack(logps).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 30 == 0 or step == 119:
        # eval
        rs = [run_episode(policy, make_task(), greedy=True)[0] for _ in range(50)]
        print(f'step={step} avgR={sum(rs)/len(rs):.3f}')
print('Key: Step penalties + terminal success teach when to call vs answer.')


## 3. 非法调用与恢复

对错误工具施加强负奖励；鼓励观察错误后改换工具。


In [ ]:
# Probe learned behavior on calc vs search tasks
for typ in ['calc', 'search']:
    task = make_task()
    while task['type'] != typ:
        task = make_task()
    # action probs at start
    typ_id = torch.tensor([0 if typ == 'calc' else 1])
    logits = policy(typ_id, torch.zeros(1, 1), torch.zeros(1, 1))
    probs = F.softmax(logits, dim=-1)[0]
    print(typ, 'action_probs', dict(zip(TOOLS, [round(p, 3) for p in probs.tolist()])))
print('Key: Inspect action marginals per task type to catch compulsive tool calling.')


## 4. 与 SFT / 过程奖励组合

工业配方常是：
1. SFT 轨迹（合法格式）  
2. 可验证 RL（成功/成本）  
3. 安全护栏（禁止危险工具参数）


In [ ]:
recipe = {
    'stage1_sft': 'tool JSON + successful traces',
    'stage2_rl': 'verifiable success - lambda*steps - illegal',
    'stage3_safety': 'firewall + schema + human review for write tools',
}
print('=== Production Recipe ===')
for k, v in recipe.items():
    print(f'{k}: {v}')
print('Key: Format competence from SFT; decision competence from RL; safety from gates.')


## 课后思考题

1. 多工具、长地平线任务如何做信用分配？
2. 环境不稳定（搜索结果变）时奖励方差怎么降？
3. 何时该 clarify 而不是瞎调工具？奖励如何塑造？
4. 工具 RL 与一般对话对齐如何共用同一底座而不互相干扰？

---
> 本节是工具调用强化学习的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
